# Week 1 deep dive: manually separate prefill and decode

This notebook is the practical bridge between `practical-baseline.ipynb` and streaming metrics. Run it after the baseline notebook. It uses the same Qwen model but calls the model forward pass directly so you can see the KV cache enter prefill and be reused in decode.

**Sources and attribution:** see `../NOTEBOOK-SOURCES.md`. The workflow is original for this lab, inspired by the learning pattern in Abi Aryan's Class 1 notebook and the Week 1 reading plan in `ROADMAP.md`.

**Success condition:** run every code cell, then explain why a longer prompt mostly changes prefill while extra output tokens add decode iterations.

In [1]:
import platform, statistics, time
from dataclasses import dataclass

import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float16 if DEVICE in {'mps', 'cuda'} else torch.float32

def synchronize():
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    elif DEVICE == 'mps':
        torch.mps.synchronize()

print({'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': DEVICE})

{'python': '3.12.13', 'torch': '2.14.0', 'transformers': '5.17.0', 'device': 'mps'}


## 1. Load once, then reuse

This load is intentionally outside every measurement. Model loading is cold-start work, not prefill or decode work.

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE).eval()
print('loaded', MODEL_ID, 'on', DEVICE)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

loaded Qwen/Qwen2.5-0.5B-Instruct on mps


## 2. Prefill: one forward pass over the whole prompt

The prompt is first formatted with Qwen's chat template. The forward pass produces logits for the first answer token and a KV cache for all prompt positions.

In [3]:
def make_inputs(text):
    messages = [{'role': 'user', 'content': text}]
    encoded = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt')
    return {key: value.to(DEVICE) for key, value in encoded.items()}

@dataclass
class PrefillResult:
    prompt_tokens: int
    prefill_seconds: float
    first_token_id: int
    cache: object

@torch.inference_mode()
def prefill_once(text):
    inputs = make_inputs(text)
    synchronize()
    started = time.perf_counter()
    outputs = model(**inputs, use_cache=True)
    synchronize()
    elapsed = time.perf_counter() - started
    cache = outputs.past_key_values
    assert cache is not None, 'Expected a KV cache when use_cache=True'
    first_token = int(torch.argmax(outputs.logits[:, -1, :], dim=-1).item())
    return PrefillResult(int(inputs['input_ids'].shape[-1]), elapsed, first_token, cache)

prompt = 'Explain caching in one sentence.'
prefill = prefill_once(prompt)
print({'prompt_tokens': prefill.prompt_tokens, 'prefill_seconds': prefill.prefill_seconds, 'first_piece': tokenizer.decode([prefill.first_token_id], skip_special_tokens=True), 'cache_type': type(prefill.cache).__name__})

{'prompt_tokens': 36, 'prefill_seconds': 0.13122950000251876, 'first_piece': 'C', 'cache_type': 'DynamicCache'}


## 3. Decode: one new token per forward pass, reusing that cache

The first token came from the prefill logits. Each later step feeds only the latest token plus `past_key_values`. That is the concrete meaning of cached decode.

In [4]:
@torch.inference_mode()
def decode_steps(first_token_id, cache, steps=8):
    token = torch.tensor([[first_token_id]], device=DEVICE)
    generated = [first_token_id]
    step_seconds = []
    current_cache = cache
    for step in range(steps - 1):
        synchronize()
        started = time.perf_counter()
        outputs = model(input_ids=token, past_key_values=current_cache, use_cache=True)
        synchronize()
        step_seconds.append(time.perf_counter() - started)
        current_cache = outputs.past_key_values
        token = torch.argmax(outputs.logits[:, -1, :], dim=-1).unsqueeze(1)
        generated.append(int(token.item()))
    return generated, step_seconds

token_ids, decode_seconds = decode_steps(prefill.first_token_id, prefill.cache, steps=8)
print('pieces:', [tokenizer.decode([token_id], skip_special_tokens=True) for token_id in token_ids])
print({'decode_steps': len(decode_seconds), 'decode_median_seconds': statistics.median(decode_seconds), 'text': tokenizer.decode(token_ids, skip_special_tokens=True)})

pieces: ['C', 'aching', ' is', ' a', ' technique', ' that', ' stores', ' frequently']
{'decode_steps': 7, 'decode_median_seconds': 0.008212249995267484, 'text': 'Caching is a technique that stores frequently'}


## 4. Practical comparison: short versus long prompt

Keep the model, device, and decode length fixed. Change only prompt length. The result is evidence for a hypothesis, not a universal benchmark claim.

In [5]:
short_prompt = 'Explain caching in one sentence.'
long_prompt = ' '.join([short_prompt] * 35)

rows = []
for label, text in [('short', short_prompt), ('long', long_prompt)]:
    result = prefill_once(text)
    _, decode_times = decode_steps(result.first_token_id, result.cache, steps=8)
    rows.append({'label': label, 'prompt_tokens': result.prompt_tokens, 'prefill_seconds': result.prefill_seconds, 'decode_median_seconds': statistics.median(decode_times)})
rows

[{'label': 'short',
  'prompt_tokens': 36,
  'prefill_seconds': 0.010830957995494828,
  'decode_median_seconds': 0.00810254200041527},
 {'label': 'long',
  'prompt_tokens': 240,
  'prefill_seconds': 0.32218412500515115,
  'decode_median_seconds': 0.008299916997202672}]

## 5. Your experiment

1. Change `long_prompt` from 35 repeats to 70 repeats.
2. Run the comparison cell again.
3. Record whether `prompt_tokens` and `prefill_seconds` changed.
4. Do **not** claim the decode median must stay identical; hardware noise and growing cache length can affect it.

**Reflection:** Which number is closest to the start of TTFT, and what extra work happens before the client sees the first streamed chunk?